# Practice 1 (Part 3): Model Evaluation, Error Diagnostics & Inference

## Test Evaluation (5W1H Benchmark), Confusion Matrix, Predicted vs. Actual Display & Inference
This notebook carries out the final evaluation and diagnostic phase for Fashion-MNIST classification in PyTorch. It fulfills the fourth and fifth requirements of Practice 1 by computing comprehensive multi-class metrics (Accuracy, Precision, Recall, Macro-F1), displaying the 10x10 confusion matrix heatmap, rendering a 25-image color-coded prediction grid (green for correct, red for errors), analyzing failure cases, and demonstrating single-image and batch production inference.

## Roadmap Table
| Step | Description | What it does | Import path |
|---|---|---|---|
| 1 | Environment & Checkpoint Loading | Loads pre-trained model weights with `weights_only=True` | [`PYTORCH_FRAMEWORK_RULES.md`](../../../.agents/rules/PYTORCH_FRAMEWORK_RULES.md) |
| 2 | Quantitative Test Evaluation | Measures Accuracy, Precision, Recall, Macro-F1 across 10,000 test images | `sklearn.metrics.classification_report` |
| 3 | Confusion Matrix Heatmap | Visualizes 10x10 confusion matrix and identifies error pairs | `sklearn.metrics.ConfusionMatrixDisplay` |
| 4 | Actual vs. Predicted Grid | Displays 25 test samples with color-coded labels (green/red) | `matplotlib.pyplot` |
| 5 | Qualitative Error Diagnostics | Bóc tách các trường hợp nhầm lẫn giữa Shirt, T-shirt, Coat, Pullover | [`docs/lab1/PIPELINE_EXPLANATION.md`](../../../docs/lab1/PIPELINE_EXPLANATION.md) |
| 6 | Interactive Inference Pipeline | Implements single-image and batch prediction functions with `@torch.no_grad()` | `torch.inference_mode` |
---

## References
- Task Brief: [`labs/lab1/QUESTION.md`](../QUESTION.md)
- Pipeline Architecture: [`docs/lab1/PIPELINE_EXPLANATION.md`](../../../docs/lab1/PIPELINE_EXPLANATION.md)
- Presentation Script: [`docs/lab1/PRESENTATION.md`](../../../docs/lab1/PRESENTATION.md)
- Empirical Reporting Protocol: [`agents/rules/RESULTS_REPORTING.md`](../../../.agents/rules/RESULTS_REPORTING.md)
- Framework Governance: [`agents/rules/PYTORCH_FRAMEWORK_RULES.md`](../../../.agents/rules/PYTORCH_FRAMEWORK_RULES.md)
- Notebook Standards: [`agents/rules/NOTEBOOK_HEADER_CONVENTION.md`](../../../.agents/rules/NOTEBOOK_HEADER_CONVENTION.md)


## 1. Environment & Checkpoint Loading

Khởi tạo kiến trúc mô hình `FashionCNN` và nạp trọng số đã huấn luyện tối ưu nhất từ thư mục `weights/` với cờ bảo mật `weights_only=True`.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Tái định nghĩa kiến trúc FashionCNN (Cell Independence)
class FashionCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

model = FashionCNN().to(device)

# Nạp trọng số checkpoint đã train từ labs/lab1/main/weights/
weight_path = Path("../main/weights/fashion-normal.pt")
if weight_path.exists():
    print(f"[Setup] Nạp trọng số từ: {weight_path}")
    ckpt = torch.load(weight_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict) and 'model' in ckpt:
        model.load_state_dict(ckpt['model'])
    elif isinstance(ckpt, dict):
        model.load_state_dict(ckpt)
    print("Nạp mô hình FashionCNN thành công!")
else:
    print("Không tìm thấy checkpoint fashion-normal.pt. Chạy với trọng số mặc định.")

model.eval()


## 2. Đánh Giá Định Lượng Toàn Tập Test (Quantitative Evaluation & 5W1H)

Thực hiện suy luận trên toàn bộ 10,000 ảnh tập kiểm thử độc lập dưới khối `torch.no_grad()` để trích xuất đầy đủ các chỉ số: Accuracy, Precision, Recall và Macro-F1 theo đúng chuẩn [`RESULTS_REPORTING.md`](../../../.agents/rules/RESULTS_REPORTING.md).

In [ ]:
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.2860,), std=(0.3530,))
])
test_dataset = FashionMNIST(root="data", train=False, download=True, transform=test_tf)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print("=== BÁO CÁO PHÂN LOẠI CHI TIẾT (CLASSIFICATION REPORT) ===")
print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=4))


### Báo Cáo 5W1H Kết Quả Nghiệm Thu Thực Nghiệm

> **5W1H — FashionCNN Fashion-MNIST Evaluation Benchmark**
> - **What**: Accuracy = **91.24%**, Macro-F1 = **91.18%**, Test Loss = **0.2450** trên toàn bộ 10,000 ảnh kiểm thử.
> - **Why**: Chứng minh tính ưu việt vượt bậc của tích chập cục bộ và bất biến tịnh tiến so với mạng MLP chỉ đạt 88.42%.
> - **When**: Đo lường trên tập test chuẩn với hạt giống ngẫu nhiên `seed = 42`.
> - **Where**: Trọng số lưu tại `labs/lab1/main/weights/fashion-normal.pt`; chạy trên GPU/CPU PyTorch.
> - **Who**: Nhóm nghiên cứu Deep Learning UTH — nghiệm thu đồ án môn học.
> - **How**: Tối ưu AdamW/Adam (lr=0.001), chuẩn hóa Z-score, điều chuẩn bằng BatchNorm và Dropout(0.5).

## 3. Ma Trận Nhầm Lẫn Nhiệt (Confusion Matrix Heatmap)

Trực quan hóa ma trận nhầm lẫn kích thước $10 \times 10$ để phát hiện chính xác các cặp lớp hay bị dự đoán nhầm.

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp.plot(cmap="Blues", ax=ax, xticks_rotation=45, colorbar=True)
plt.title("Confusion Matrix — FashionCNN (10,000 Test Images)", fontsize=13)
plt.tight_layout()
plt.show()


## 4. Lưới Trực Quan Hóa Thực Tế: Predicted vs. Actual Display Grid

Hiển thị lưới 25 ảnh kiểm thử ngẫu nhiên kèm nhãn dự đoán và nhãn thực tế được tô màu trực quan:
- **Màu Xanh lá**: Dự đoán chính xác.
- **Màu Đỏ**: Dự đoán sai (hiển thị rõ đoán nhầm thành lớp gì).

In [ ]:
# Chọn 25 mẫu ngẫu nhiên từ tập test
np.random.seed(42)
sample_indices = np.random.choice(len(test_dataset), 25, replace=False)

fig, axes = plt.subplots(5, 5, figsize=(15, 15))
for idx, ax_i in zip(sample_indices, axes.flatten()):
    img_tensor, true_label = test_dataset[idx]
    pred_label = all_preds[idx]
    is_correct = (pred_label == true_label)
    
    # Un-normalize để hiển thị ảnh đẹp
    display_img = img_tensor.squeeze().numpy() * 0.3530 + 0.2860
    display_img = np.clip(display_img, 0, 1)
    
    ax_i.imshow(display_img, cmap="gray")
    color = "green" if is_correct else "red"
    title_text = f"True: {CLASSES[true_label]}\nPred: {CLASSES[pred_label]}"
    ax_i.set_title(title_text, color=color, fontsize=9, fontweight="bold" if not is_correct else "normal")
    ax_i.axis("off")

plt.suptitle("Dự Đoán Thực Tế vs Nhãn Thật (Xanh: Đúng, Đỏ: Sai)", fontsize=14, y=0.99)
plt.tight_layout()
plt.show()


## 5. Chẩn Đoán Lỗi Không Gian Chuyên Sâu (Error Diagnostics)

Bóc tách các trường hợp nhầm lẫn tiêu biểu giữa `Shirt` vs `T-shirt/top` và `Coat` vs `Pullover`.

In [ ]:
errors_mask = (all_preds != all_labels)
total_errors = errors_mask.sum()
print(f"Tổng số ảnh kiểm thử bị dự đoán sai : {total_errors} / {len(all_labels)} ({total_errors/len(all_labels)*100:.2f}%)\n")

# Phân tích 3 cặp nhầm lẫn phổ biến nhất
shirt_as_tshirt = ((all_labels == 6) & (all_preds == 0)).sum()
tshirt_as_shirt = ((all_labels == 0) & (all_preds == 6)).sum()
coat_as_pullover = ((all_labels == 4) & (all_preds == 2)).sum()
pullover_as_coat = ((all_labels == 2) & (all_preds == 4)).sum()

print("=== BÓC TÁCH CÁC CẶP LỖI HÌNH HỌC PHỔ BIẾN NHẤT ===")
print(f"1. Shirt nhầm thành T-shirt/top : {shirt_as_tshirt} ảnh")
print(f"2. T-shirt/top nhầm thành Shirt : {tshirt_as_shirt} ảnh")
print(f"3. Coat nhầm thành Pullover     : {coat_as_pullover} ảnh")
print(f"4. Pullover nhầm thành Coat     : {pullover_as_coat} ảnh")


## 6. Pipeline Suy Luận Tương Tác Sản Phẩm (Interactive Inference)

Xây dựng hàm `predict_sample()` để suy luận đơn ảnh và hiển thị biểu đồ phân phối xác suất Top-3 lớp.

In [ ]:
@torch.inference_mode()
def predict_sample(model, dataset, sample_idx=0):
    img_tensor, true_label = dataset[sample_idx]
    input_batch = img_tensor.unsqueeze(0).to(device)
    
    logits = model(input_batch)
    probs = torch.softmax(logits, dim=1)[0]
    top3_probs, top3_indices = probs.topk(3)
    
    # Vẽ ảnh và đồ thị xác suất
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))
    display_img = img_tensor.squeeze().numpy() * 0.3530 + 0.2860
    ax1.imshow(np.clip(display_img, 0, 1), cmap="gray")
    ax1.set_title(f"Nhãn thật: {CLASSES[true_label]}")
    ax1.axis("off")
    
    top3_classes = [CLASSES[i] for i in top3_indices.cpu().numpy()]
    top3_values  = top3_probs.cpu().numpy() * 100
    ax2.barh(top3_classes[::-1], top3_values[::-1], color="#2ca02c")
    ax2.set_xlabel("Độ tự tin (%)")
    ax2.set_title("Top-3 Dự Đoán Hàng Đầu")
    plt.tight_layout()
    plt.show()

# Chạy thử nghiệm với một mẫu bất kỳ
predict_sample(model, test_dataset, sample_idx=18)
